# Diabetic Retinopathy Screening Agent
Vicheda Narith, Maanvi Sarwadi

## Setup & Dependencies

Run the following script to load packages and dependencies

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import numpy as np
import requests
import os
from tqdm import tqdm

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

import os
# Check the checkpoint is there
# checkpoint_path = f"/content/drive/MyDrive/3rd Year: 2025-2026/Spring Quarter/CS 496: Agent AI/final_project/checkpoints/dr_best.pth"
os_checkpoint_path = "checkpoints/dr_best.pth"
print(os.path.exists(os_checkpoint_path))  # should print True

True


In [3]:
UNCERTAINTY_THRESHOLD = 0.5
DR_GRADES = {
    0: "No apparent DR",
    1: "Mild NPDR",
    2: "Moderate NPDR",
    3: "Severe NPDR",
    4: "Proliferative DR"
}

## Load Pre-trained Models

In [ ]:
# transforms to apply images
def get_transforms():
    return transforms.Compose(
        [
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]
    )

In [ ]:
from datasets import load_dataset
from torch.utils.data import random_split

# Load APTOS with labels — already 224x224, no preprocessing needed
aptos = load_dataset("bumbledeep/aptos", split="train")

print(aptos)
print(aptos.features)
print(aptos[0].keys())  # should show: image, label_code, label

Dataset({
    features: ['image', 'label_code', 'label'],
    num_rows: 3662
})
{'image': Image(mode=None, decode=True), 'label_code': Value('int64'), 'label': Value('string')}
dict_keys(['image', 'label_code', 'label'])


In [8]:
class DRDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data      = data
        self.transform = transform or get_transforms()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row   = self.data[idx]
        img   = row['image'].convert('RGB')
        label = row['label_code']  # integer 0–4
        if self.transform:
            img = self.transform(img)
        return img, label


# Train/val split (85/15)
n       = len(aptos)
n_train = int(0.85 * n)
n_val   = n - n_train

train_data, val_data = random_split(aptos, [n_train, n_val],
                                     generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(DRDataset(train_data), batch_size=8, shuffle=True,  num_workers=2)
val_loader   = DataLoader(DRDataset(val_data),   batch_size=8, shuffle=False, num_workers=2)

print(f'Train: {n_train} | Val: {n_val}')

Train: 3112 | Val: 550


In [9]:
import timm

class DRVisionModel(nn.Module):
    def __init__(self, num_classes=5, dropout_rate=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b0',
            pretrained=True,
            num_classes=0  # remove default head
        )
        feature_dim = self.backbone.num_features  # 1536

        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.backbone(x))

    def enable_mc_dropout(self):
        for m in self.modules():
            if isinstance(m, nn.Dropout):
                m.train()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = DRVisionModel().to(DEVICE)
model.load_state_dict(torch.load(os_checkpoint_path, map_location=DEVICE))
model.eval()
print('Weights loaded.')

print(f'Device: {DEVICE}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

Weights loaded.
Device: cpu
Parameters: 4,665,985


/var/folders/7p/ypnwq5vj3t70728pfnqzb5940000gn/T/ipykernel_16494/3223390465.py:31: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(os_checkpoi

In [10]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# Clear any cached memory from previous runs
torch.cuda.empty_cache()

In [ ]:
from collections import Counter
labels = [aptos[i]['label_code'] for i in range(len(aptos))]
dist = Counter(labels)
print(dist)

In [ ]:
import matplotlib.pyplot as plt

def plot_training_history(train_losses, val_losses):
    plt.figure(figsize=(8, 5))
    plt.plot(train_losses, 'o-', color='steelblue', label='Train Loss')
    plt.plot(val_losses,   'o-', color='coral',     label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [13]:
# Monte Carlo Dropout Inference

def mc_dropout_predict(model, image_tensor, n_passes=30):
    """
    Run n_passes stochastic forward passes with dropout active.
    Returns grade prediction + uncertainty scores.
    """
    model.eval()
    model.enable_mc_dropout()  # keep dropout on during inference

    image_tensor = image_tensor.to(DEVICE)
    all_probs = []

    with torch.no_grad():
        for _ in range(n_passes):
            logits = model(image_tensor)
            probs  = F.softmax(logits, dim=1)
            all_probs.append(probs.cpu().numpy())

    all_probs  = np.array(all_probs).squeeze()
    probs_mean = all_probs.mean(axis=0)
    all_preds  = np.argmax(all_probs, axis=1)

    return {
        'grade':      int(np.argmax(probs_mean)),
        'grade_label': DR_GRADES[int(np.argmax(probs_mean))],
        'confidence': float(probs_mean.max()),
        'variance':   float(np.var(all_preds)),
        'entropy':    float(-np.sum(probs_mean * np.log(probs_mean + 1e-8))),
        'probs_mean': probs_mean.tolist(),
        'all_preds':  all_preds.tolist()
    }

print('MC Dropout defined.')

MC Dropout defined.


In [14]:
# Validate Image
import cv2

def check_image_quality(img: Image.Image) -> dict:
    """
    Checks blur and field-of-view before grading.
    Rejects images that are too blurry or poorly framed.
    """
    img_gray = np.array(img.convert('L'))

    # Blur detection via Laplacian variance
    blur_score = cv2.Laplacian(img_gray, cv2.CV_64F).var()
    if blur_score < 50:
        return {'valid': False, 'reason': f'Image too blurry (score: {blur_score:.1f})'}

    # Field of view: retinal disc should fill >15% of frame
    non_black = np.sum(img_gray > 10) / img_gray.size
    if non_black < 0.15:
        return {'valid': False, 'reason': f'Insufficient field of view ({non_black:.1%})'}

    return {'valid': True, 'reason': 'OK'}

print('Image validation defined.')

Image validation defined.


In [15]:
import faiss

# Clinical knowledge base
KNOWLEDGE_BASE = [
    {
        'id': 'aao_0',
        'source': 'AAO Guidelines 2023',
        'text': 'Grade 0 (No DR): Annual dilated eye exam. No treatment required. Optimize glycemic control.'
    },
    {
        'id': 'aao_1',
        'source': 'AAO Guidelines 2023',
        'text': 'Grade 1 (Mild NPDR): Microaneurysms only. Follow-up in 12 months. No ocular treatment needed.'
    },
    {
        'id': 'aao_2',
        'source': 'AAO Guidelines 2023',
        'text': 'Grade 2 (Moderate NPDR): Refer to ophthalmologist within 3-6 months. Consider anti-VEGF if macular edema present.'
    },
    {
        'id': 'aao_3',
        'source': 'AAO Guidelines 2023',
        'text': 'Grade 3 (Severe NPDR): Urgent referral within 1 month. High risk of progression to PDR. Panretinal photocoagulation should be considered.'
    },
    {
        'id': 'aao_4',
        'source': 'AAO Guidelines 2023',
        'text': 'Grade 4 (Proliferative DR): Emergency referral to vitreoretinal surgeon. Immediate treatment indicated. Risk of irreversible vision loss.'
    },
    {
        'id': 'pub_1',
        'source': 'Gulshan et al. JAMA 2016',
        'text': 'Deep learning achieved AUC 0.991 for referable DR detection. Sensitivity 97.5%, specificity 93.4% on EyePACS validation set.'
    },
    {
        'id': 'pub_2',
        'source': 'Nature Medicine 2022',
        'text': 'Calibrated uncertainty estimates are critical for safe AI in clinical settings. Monte Carlo Dropout is a validated approach for uncertainty quantification in medical imaging.'
    }
]

def build_faiss_index(knowledge_base, dim=128):
    """Build FAISS index from knowledge base using simple character embeddings."""
    def embed(text):
        vec = np.zeros(dim)
        for char in text.lower():
            vec[ord(char) % dim] += 1
        norm = np.linalg.norm(vec)
        return (vec / norm).astype('float32') if norm > 0 else vec.astype('float32')

    embeddings = np.array([embed(doc['text']) for doc in knowledge_base])
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)
    return index, embed  # return embed fn so we can reuse it

faiss_index, embed_fn = build_faiss_index(KNOWLEDGE_BASE)
print(f'FAISS index built: {faiss_index.ntotal} documents')


def retrieve(query, top_k=3):
    """Retrieve top-k relevant documents for a query."""
    query_vec = embed_fn(query).reshape(1, -1)
    distances, indices = faiss_index.search(query_vec, top_k)
    return [KNOWLEDGE_BASE[i] for i in indices[0] if i < len(KNOWLEDGE_BASE)]


def generate_referral_report(mc_result, patient_id='UNKNOWN'):
    """
    Template-based referral report — no API call needed.
    Retrieves relevant guidelines from FAISS and formats them directly.
    """
    query    = f"DR grade {mc_result['grade']} {mc_result['grade_label']} referral treatment"
    docs     = retrieve(query, top_k=2)
    guidance = '\n'.join([f"- [{d['source']}]: {d['text']}" for d in docs])

    referral_reason = 'high model uncertainty' if mc_result['variance'] > UNCERTAINTY_THRESHOLD \
                      else 'grade severity'

    return f"""
CLINICAL REPORT — Patient: {patient_id}
{'='*50}

1. DIAGNOSIS SUMMARY
   DR Grade: {mc_result['grade']} — {mc_result['grade_label']}
   Confidence: {mc_result['confidence']:.2%}

2. SEVERITY ASSESSMENT
   {'Referable DR detected. Specialist review required.' if mc_result['grade'] >= 2
    else 'Non-referable DR. Routine monitoring recommended.'}

3. REFERRAL RECOMMENDATION
   Triggered by: {referral_reason}
   {'URGENT referral to ophthalmologist required.' if mc_result['grade'] >= 3
    else 'Referral to ophthalmologist within 3-6 months.'}

4. RECOMMENDED NEXT STEPS
{guidance}

5. UNCERTAINTY NOTE
   Model variance: {mc_result['variance']:.4f} (threshold: {UNCERTAINTY_THRESHOLD})
   Entropy: {mc_result['entropy']:.4f}
   {'⚠ High uncertainty — human review strongly recommended.'
    if mc_result['variance'] > UNCERTAINTY_THRESHOLD
    else '✓ Confidence within acceptable range.'}
""".strip()

print('RAG pipeline defined.')

FAISS index built: 7 documents
RAG pipeline defined.


In [ ]:
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def run_dr_agent(image: Image.Image, patient_id='UNKNOWN', n_passes=30, verbose=True):
    """
    Full DR Screening Agent.

    State:  retinal image + model confidence score
    Actions: DIAGNOSE or REFER
    Policy:  high uncertainty OR high grade → REFER
             otherwise → DIAGNOSE
    """
    if verbose:
        print(f'\n{"="*50}')
        print(f'DR AGENT — Patient: {patient_id}')
        print(f'{"="*50}')

    # Validate image
    if verbose: print('\n[1] Validating image quality...')
    quality = check_image_quality(image)
    if not quality['valid']:
        if verbose: print(f'    ✗ Rejected: {quality["reason"]}')
        return {'status': 'REJECTED', 'reason': quality['reason']}
    if verbose: print('    ✓ Image OK')

    # Vision model & uncertainty
    if verbose: print(f'\n[2] Running vision model ({n_passes} MC passes)...')
    img_tensor = val_transform(image).unsqueeze(0)
    mc_result  = mc_dropout_predict(model, img_tensor, n_passes=n_passes)

    if verbose:
        print(f'    Grade:      {mc_result["grade"]} — {mc_result["grade_label"]}')
        print(f'    Confidence: {mc_result["confidence"]:.2%}')
        print(f'    Variance:   {mc_result["variance"]:.4f}')

    # Decision policy
    high_uncertainty = mc_result['variance'] > UNCERTAINTY_THRESHOLD
    high_severity    = mc_result['grade'] >= 2
    needs_referral   = high_uncertainty or high_severity

    if verbose:
        print(f'\n[3] Decision policy:')
        print(f'    High uncertainty: {high_uncertainty}')
        print(f'    High severity:    {high_severity}')
        print(f'    → Action: {"REFER" if needs_referral else "DIAGNOSE"}')

    # Generate report
    if needs_referral:
        if verbose: print('\n[4] Generating RAG referral report...')
        report = generate_referral_report(mc_result, patient_id)
        action = 'REFER'
    else:
        if verbose: print('\n[4] Generating automated diagnosis...')
        report = (
            f'AUTOMATED DIAGNOSIS\n'
            f'Patient: {patient_id}\n'
            f'DR Grade: {mc_result["grade"]} ({mc_result["grade_label"]})\n'
            f'Confidence: {mc_result["confidence"]:.2%}\n'
            f'Recommendation: Annual follow-up. No referral required.'
        )
        action = 'DIAGNOSE'

    if verbose:
        print(f'\n{"─"*50}')
        print(report)

    return {
        'status':    'PROCESSED',
        'action':    action,
        'mc_result': mc_result,
        'report':    report
    }

print('Agent loop defined.')

Agent loop defined.


In [ ]:
# Evaluation

# Load your trained checkpoint
model.load_state_dict(torch.load(os_checkpoint_path, map_location=DEVICE))

# Grab one image from val set
# sample     = val_data[0]
# image      = sample['image'] if isinstance(sample, dict) else aptos[val_data.indices[0]]['image']
# image      = image.convert('RGB')

# Get a raw image from APTOS directly (before transforms)
raw_sample = aptos[val_data.indices[0]]
image      = raw_sample['image'].convert('RGB')

# Run the agent
result = run_dr_agent(image, patient_id='PT-001', n_passes=30)


# # Run the agent
# result = run_dr_agent(image, patient_id='PT-001', n_passes=30)


DR AGENT — Patient: PT-001

[1] Validating image quality...
    ✓ Image OK

[2] Running vision model (30 MC passes)...


/var/folders/7p/ypnwq5vj3t70728pfnqzb5940000gn/T/ipykernel_16494/2055406146.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(os_checkpoin

    Grade:      4 — Proliferative DR
    Confidence: 28.47%
    Variance:   0.2722

[3] Decision policy:
    High uncertainty: False
    High severity:    True
    → Action: REFER

[4] Generating RAG referral report...

──────────────────────────────────────────────────
CLINICAL REPORT — Patient: PT-001

1. DIAGNOSIS SUMMARY
   DR Grade: 4 — Proliferative DR
   Confidence: 28.47%

2. SEVERITY ASSESSMENT
   Referable DR detected. Specialist review required.

3. REFERRAL RECOMMENDATION
   Triggered by: grade severity
   URGENT referral to ophthalmologist required.

4. RECOMMENDED NEXT STEPS
- [AAO Guidelines 2023]: Grade 4 (Proliferative DR): Emergency referral to vitreoretinal surgeon. Immediate treatment indicated. Risk of irreversible vision loss.
- [AAO Guidelines 2023]: Grade 2 (Moderate NPDR): Refer to ophthalmologist within 3-6 months. Consider anti-VEGF if macular edema present.

5. UNCERTAINTY NOTE
   Model variance: 0.2722 (threshold: 0.5)
   Entropy: 1.5438
   ✓ Confidence wit

In [19]:
from sklearn.metrics import roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt

model.eval()
all_preds, all_labels, all_conf = [], [], []

# Run on subset of val set (100 images is fine)
for idx in val_data.indices[:100]:
    raw   = aptos[idx]
    image = raw['image'].convert('RGB')
    label = raw['label_code']

    img_tensor = val_transform(image).unsqueeze(0)
    mc         = mc_dropout_predict(model, img_tensor, n_passes=20)

    all_preds.append(mc['grade'])
    all_labels.append(label)
    all_conf.append(mc['confidence'])

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

# Accuracy
accuracy = (all_preds == all_labels).mean()
print(f'Accuracy:    {accuracy:.4f}')

# Binary: referable (>=2) vs not
pred_ref = (all_preds  >= 2).astype(int)
true_ref = (all_labels >= 2).astype(int)

TP = np.sum((pred_ref == 1) & (true_ref == 1))
TN = np.sum((pred_ref == 0) & (true_ref == 0))
FP = np.sum((pred_ref == 1) & (true_ref == 0))
FN = np.sum((pred_ref == 0) & (true_ref == 1))

sensitivity = TP / (TP + FN + 1e-8)
specificity = TN / (TN + FP + 1e-8)
print(f'Sensitivity: {sensitivity:.4f}')
print(f'Specificity: {specificity:.4f}')

# AUC
if len(np.unique(true_ref)) > 1:
    all_referable_probs = []

    for idx in val_data.indices[:100]:
        raw        = aptos[idx]
        image      = raw['image'].convert('RGB')
        img_tensor = val_transform(image).unsqueeze(0)
        mc         = mc_dropout_predict(model, img_tensor, n_passes=20)
        ref_prob   = sum(mc['probs_mean'][2:])
        all_referable_probs.append(ref_prob)

    auc = roc_auc_score(true_ref, np.array(all_referable_probs))
    print(f'AUC-ROC:     {auc:.4f}')

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6,5))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xlabel('Predicted Grade')
plt.ylabel('True Grade')
plt.title('Confusion Matrix')
plt.xticks(range(5), DR_GRADES.values(), rotation=20)
plt.yticks(range(5), DR_GRADES.values())
plt.tight_layout()
plt.show()

KeyboardInterrupt: 